## Imports

In [17]:
import re
import os
import pickle
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [18]:
with open("data/tech_knowledge_base.pkl", "rb") as f:
    tech_knowledge_base = pickle.load(f)

## Chunking methods

In [19]:
def simple_chunking(text, chunk_size=2000, chunk_overlap=1000):
    idx = 0
    result = []
    while idx < len(text):
        end_idx = min(idx + chunk_size, len(text))
        result.append(text[idx : end_idx])
        idx += chunk_size - chunk_overlap
    return result

In [20]:
def paragraph_chunking(text):
    return re.split(r"\n\s*\n", text.strip())

In [21]:
def section_chunking(text, level=2):
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    
    parts = pattern.split(text)
    
    sections = []
    
    for i in range(1, len(parts),3):
        header = parts[i] + parts[i+1] # "## " + "Title"
        header = header.strip()
        
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()
        
        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    return sections

In [22]:
def llm(prompt, model='gpt-4o-mini'):
    openai_client = OpenAI()
    messages = [
        # {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': prompt}
    ]
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

In [23]:
def intelligent_chunking(text, prompt_template):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [24]:
def chunk_documents(documents, chunk_strategy, **kwargs):
    final_chunks = []
    kwargs = {**kwargs}
    for doc in tqdm(documents):
        doc_copy = doc.copy()
        doc_content = doc_copy.pop("content", "Content not found")
        chunks = chunk_strategy(doc_content, **kwargs)
        chunks = [{**doc_copy, 'chunk_content': chunk} for chunk in chunks]
        final_chunks.extend(chunks)
    return final_chunks

In [25]:
prompt_template = """
    You are an expert document/text analyst with a strong grasp of logic, english, code and general comprehension.
    
    Split the provided document into logical sections that make sense for a Q&A system. Each section should be self-contained and cover a specific topic or concept.
    
    <DOCUMENT>
    {document}
    </DOCUMENT>
    
    Use this format:
    
    ## Section Name
    
    Section content with all relevant details
    
    ---
    
    ## Another Section Name
    
    Another section content
    
    ---
""".strip()

In [26]:
simple_chunks = chunk_documents(tech_knowledge_base, 
                                chunk_strategy=simple_chunking)

paragraph_chunks = chunk_documents(tech_knowledge_base, 
                                   chunk_strategy=paragraph_chunking)

section_chunks = chunk_documents(tech_knowledge_base, 
                                chunk_strategy=section_chunking)

intelligent_chunks = chunk_documents(tech_knowledge_base, 
                                    chunk_strategy=intelligent_chunking,
                                    prompt_template=prompt_template)



  0%|          | 0/19306 [00:00<?, ?it/s]

  0%|          | 0/19306 [00:00<?, ?it/s]

  0%|          | 0/19306 [00:00<?, ?it/s]

  0%|          | 0/19306 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Export

In [27]:
def export_data(obj, file_path):
    if not file_path.endswith('.pkl'):
        raise ValueError(f"File {file_path} is not a pickle file")
    
    with open(file_path, 'wb') as f:
        pickle.dump(obj, f)

In [28]:
def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} does not exist")
    
    if not file_path.endswith('.pkl'):
        raise ValueError(f"File {file_path} is not a pickle file")
    
    with open(file_path, 'rb') as f:
        return pickle.load(f)

In [29]:
export_data(simple_chunks, 'data/simple_chunks.pkl')
export_data(paragraph_chunks, 'data/paragraph_chunks.pkl')
export_data(section_chunks, 'data/section_chunks.pkl')
# export_data(intelligent_chunks, 'data/intelligent_chunks.pkl')
